# UD5.02. Una red desde cero

**Módulo 5073 · Programación de Inteligencia Artificial · Curso 2026/27**
Bloques 3, 4, 5 y 6 de los apuntes · Criterios **2.b** y **2.d**

---

Este es el cuaderno que hay que escribir, no leer. Al terminar tendrás una red neuronal
completa —ida, pérdida, vuelta y actualización— en unas sesenta líneas de NumPy, sin
TensorFlow, sin Keras y sin nada más que `@` y `np.maximum`.

El motivo de hacerlo es concreto y no es purista: **quien ha programado la
retropropagación una vez no vuelve a mirar `model.fit()` como una caja negra**, y sabe qué
tocar cuando un entrenamiento no converge. Es lo que separa ajustar hiperparámetros a
ciegas de diagnosticar.

El orden es este:

1. Demostrar que sin activación no lineal, una pila de capas es una capa.
2. Construir la pérdida y ver por qué no es la métrica.
3. Calcular un gradiente a mano y comprobarlo numéricamente.
4. Escribir el bucle entero, y entrenarlo sobre XOR.
5. Entrenarlo sobre los clientes de TechStore de la UD3.
6. Comprobar contra `tf.GradientTape` que los gradientes coinciden.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(20262027)
np.set_printoptions(precision=4, suppress=True)

print("numpy", np.__version__)

---

## 1. Sin no linealidad, cien capas son una recta

El argumento del bloque 3 en tres líneas. Si se apilan dos capas densas sin activación:

$$Z_2 = (X W_1 + b_1) W_2 + b_2 = X (W_1 W_2) + (b_1 W_2 + b_2)$$

que es una sola capa con $W' = W_1 W_2$ y $b' = b_1 W_2 + b_2$. Vamos a comprobarlo con
números en vez de creerlo.

In [ ]:
X = rng.normal(size=(6, 4))

W1 = rng.normal(size=(4, 8));  b1 = rng.normal(size=8)
W2 = rng.normal(size=(8, 3));  b2 = rng.normal(size=3)

# Dos capas, SIN activacion.
dos_capas = (X @ W1 + b1) @ W2 + b2

# Una sola capa, con los pesos combinados.
W_equiv = W1 @ W2
b_equiv = b1 @ W2 + b2
una_capa = X @ W_equiv + b_equiv

print("Diferencia maxima:", np.abs(dos_capas - una_capa).max())
assert np.allclose(dos_capas, una_capa)
print("Son la MISMA funcion. La segunda capa no ha aportado nada.")

In [ ]:
# Y con cincuenta capas pasa exactamente lo mismo.
W_acumulado = np.eye(4)
b_acumulado = np.zeros(4)
Z = X.copy()

for _ in range(50):
    W = rng.normal(size=(4, 4)) * 0.3
    b = rng.normal(size=4) * 0.1
    Z = Z @ W + b
    W_acumulado = W_acumulado @ W
    b_acumulado = b_acumulado @ W + b

print("Diferencia maxima con la capa unica equivalente:",
      np.abs(Z - (X @ W_acumulado + b_acumulado)).max())
print()
print("Cincuenta capas lineales = una capa lineal.")
print("La capacidad de una red NO esta en el numero de capas: esta en la activacion.")

> Ese es el motivo de que olvidar `activation="relu"` en una capa oculta sea un error
> silencioso: el modelo entrena, no da ningún aviso, y no puede aprender nada que no sea
> una recta.

### Las activaciones, dibujadas con su derivada

La derivada es lo que importa, porque es lo que se multiplica al retropropagar.

In [ ]:
z = np.linspace(-6, 6, 400)

activaciones = {
    "ReLU":     (np.maximum(0, z),                (z > 0).astype(float)),
    "Sigmoide": (1 / (1 + np.exp(-z)),            None),
    "Tanh":     (np.tanh(z),                      1 - np.tanh(z) ** 2),
    "LeakyReLU": (np.where(z > 0, z, 0.01 * z),   np.where(z > 0, 1.0, 0.01)),
}
s = 1 / (1 + np.exp(-z))
activaciones["Sigmoide"] = (s, s * (1 - s))

fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
for nombre, (valor, derivada) in activaciones.items():
    ejes[0].plot(z, valor, label=nombre)
    ejes[1].plot(z, derivada, label=nombre)

ejes[0].set_title("Las activaciones no cambian la escala de la entrada")
ejes[1].set_title("La derivada de la sigmoide no pasa de 0,25")
for eje, etiqueta in zip(ejes, ["f(z)", "df/dz"]):
    eje.set_xlabel("z (entrada de la activacion)")
    eje.set_ylabel(etiqueta)
    eje.axhline(0, color="0.7", lw=0.8)
    eje.axvline(0, color="0.7", lw=0.8)
    eje.legend(fontsize=8)
ejes[1].axhline(0.25, color="crimson", ls="--", lw=1)
fig.tight_layout()
plt.show()

In [ ]:
# La cuenta del desvanecimiento del gradiente, con numeros.
print(f"{'capas':>6} {'sigmoide':>12} {'tanh':>12} {'ReLU':>12}")
print("-" * 46)
for capas in (1, 5, 10, 20, 50):
    print(f"{capas:>6} {0.25 ** capas:>12.2e} {1.0 ** capas:>12.2e} {1.0 ** capas:>12.2e}")
print()
print("Con sigmoides, a las 20 capas el gradiente que llega a la primera es 10^-12:")
print("esa capa NO se entera de nada. Es lo que mantuvo las redes profundas")
print("sin funcionar durante veinte años, y ReLU es la mitad de la solucion.")

---

## 2. La pérdida, y por qué no es la métrica

La exactitud no es derivable: es un escalón. Vamos a dibujarlo, porque es el argumento
entero del bloque 4.1 y se entiende mirándolo.

In [ ]:
p = np.linspace(0.001, 0.999, 500)

# Para una muestra cuya verdad es 1:
exactitud = (p >= 0.5).astype(float)
entropia = -np.log(p)

fig, ejes = plt.subplots(1, 2, figsize=(11, 4))

ejes[0].plot(p, exactitud, color="crimson", lw=2)
ejes[0].set_title("La exactitud es un escalon: derivada 0 o indefinida")
ejes[0].set_ylabel("acierto (0 o 1)")

ejes[1].plot(p, entropia, color="steelblue", lw=2)
ejes[1].set_title("La entropia cruzada baja suave: siempre hay pendiente")
ejes[1].set_ylabel("perdida  -log(p)")
ejes[1].set_ylim(0, 7)

for eje in ejes:
    eje.set_xlabel("probabilidad que el modelo da a la clase correcta")
    eje.axvline(0.5, color="0.7", ls="--", lw=1)
fig.suptitle("Verdad = 1. Un descenso de gradiente sobre el panel izquierdo "
             "no tiene hacia donde ir", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
def entropia_cruzada_binaria(y, p, eps=1e-12):
    # El recorte evita log(0), que es -infinito y contagia NaN a todo el gradiente.
    p = np.clip(p, eps, 1 - eps)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))


casos = [
    ("acierta con seguridad",      1, 0.99),
    ("acierta con dudas",          1, 0.55),
    ("falla con dudas",            1, 0.45),
    ("falla con seguridad",        1, 0.01),
    ("acierta con seguridad (0)",  0, 0.01),
    ("falla con seguridad (0)",    0, 0.99),
]

print(f"{'caso':30} {'y':>3} {'p':>6} {'acierta':>9} {'perdida':>9}")
print("-" * 62)
for nombre, y, p_ in casos:
    acierta = int((p_ >= 0.5) == y)
    print(f"{nombre:30} {y:>3} {p_:>6.2f} {acierta:>9} "
          f"{entropia_cruzada_binaria(np.array([y]), np.array([p_])):>9.3f}")
print()
print("Para la exactitud, 'falla con dudas' y 'falla con seguridad' cuentan igual:")
print("los dos son un fallo. Para la perdida, el segundo cuesta casi seis veces")
print("mas. Esa es exactamente la informacion que la exactitud tira.")

### La pérdida inicial esperada

Antes de aprender nada, un modelo bien inicializado reparte la probabilidad por igual
entre las $k$ clases, así que su pérdida vale $\log(k)$. Es la comprobación de sensatez
más barata que existe y está en el bloque 16.

In [ ]:
print(f"{'clases':>7} {'log(k)':>9}   ejemplo")
print("-" * 44)
for k, ejemplo in [(2, "abandono de clientes"), (10, "Fashion-MNIST"),
                   (43, "señales de trafico"), (1000, "ImageNet")]:
    print(f"{k:>7} {np.log(k):>9.3f}   {ejemplo}")
print()
print("Si la primera epoca sale muy por encima: etiquetas mal, activacion de salida")
print("equivocada, o datos sin normalizar. Si sale muy por debajo: fuga de informacion.")

---

## 3. El gradiente a mano, y comprobado numéricamente

Antes de escribir la retropropagación entera conviene tener una forma **independiente** de
comprobar si un gradiente está bien. Esa forma es la definición de derivada:

$$\frac{\partial L}{\partial \theta} \approx \frac{L(\theta + \epsilon) - L(\theta - \epsilon)}{2\epsilon}$$

Es lentísima —hay que evaluar la red dos veces por parámetro— y por eso no se usa para
entrenar. Pero para comprobar es perfecta, y es lo que se hace siempre que se escribe una
capa nueva a mano.

In [ ]:
def gradiente_numerico(f, theta, eps=1e-5):
    # Diferencias centradas, parametro a parametro. Lento a proposito.
    grad = np.zeros_like(theta)
    it = np.nditer(theta, flags=["multi_index"])
    while not it.finished:
        i = it.multi_index
        original = theta[i]
        theta[i] = original + eps;  mas = f()
        theta[i] = original - eps;  menos = f()
        theta[i] = original
        grad[i] = (mas - menos) / (2 * eps)
        it.iternext()
    return grad

In [ ]:
# Una red 4 -> 3 -> 1, con datos de juguete.
X_j = rng.normal(size=(10, 4))
y_j = (rng.random((10, 1)) < 0.5).astype(float)

W1 = rng.normal(size=(4, 3)) * 0.5;  b1 = np.zeros(3)
W2 = rng.normal(size=(3, 1)) * 0.5;  b2 = np.zeros(1)


def ida():
    Z1 = X_j @ W1 + b1
    A1 = np.maximum(0, Z1)
    Z2 = A1 @ W2 + b2
    A2 = 1 / (1 + np.exp(-Z2))
    return Z1, A1, Z2, A2


def perdida():
    return entropia_cruzada_binaria(y_j, ida()[3])


# El gradiente analitico, escrito a mano.
Z1, A1, Z2, A2 = ida()
m = len(X_j)
dZ2 = (A2 - y_j) / m
dW2 = A1.T @ dZ2
db2 = dZ2.sum(axis=0)
dA1 = dZ2 @ W2.T
dZ1 = dA1 * (Z1 > 0)
dW1 = X_j.T @ dZ1
db1 = dZ1.sum(axis=0)

print(f"{'parametro':>10} {'analitico':>28} {'error relativo':>16}")
print("-" * 58)
for nombre, theta, analitico in [("W1", W1, dW1), ("b1", b1, db1),
                                 ("W2", W2, dW2), ("b2", b2, db2)]:
    numerico = gradiente_numerico(perdida, theta)
    error = (np.abs(analitico - numerico).max()
             / max(np.abs(analitico).max(), np.abs(numerico).max(), 1e-12))
    print(f"{nombre:>10} {str(analitico.shape):>28} {error:>16.2e}")
print()
print("Errores del orden de 1e-8 o menores: la retropropagacion esta bien.")
print("Por encima de 1e-4 hay un fallo, y casi siempre esta en la derivada")
print("de la activacion o en un .T que falta.")

### Por qué `dZ2 = A2 - y`

No es una casualidad ni una simplificación de los apuntes. La derivada de la entropía
cruzada respecto de $\hat{p}$ es $\frac{\hat{p}-y}{\hat{p}(1-\hat{p})}$, y la derivada de
la sigmoide respecto de $z$ es $\hat{p}(1-\hat{p})$. Al multiplicarlas, el denominador se
cancela con el segundo factor y queda $\hat{p} - y$.

Esa cancelación es el motivo de que **sigmoide y entropía cruzada binaria vayan siempre
juntas**, y de que `softmax` y `categorical_crossentropy` también: no es solo que tengan
sentido estadístico, es que su gradiente combinado es limpio y numéricamente estable.

In [ ]:
# Comprobacion de la cancelacion, con numeros.
p_ = np.array([0.2, 0.7, 0.9])
y_ = np.array([0.0, 1.0, 1.0])

d_perdida = (p_ - y_) / (p_ * (1 - p_))    # dL/dp
d_sigmoide = p_ * (1 - p_)                 # dp/dz

print("dL/dp          ", (d_perdida).round(4))
print("dp/dz          ", (d_sigmoide).round(4))
print("producto (dL/dz)", (d_perdida * d_sigmoide).round(4))
print("p - y           ", (p_ - y_).round(4))

---

## 4. El bucle de entrenamiento, entero

Ahora sí. Sesenta líneas y una red neuronal completa.

In [ ]:
def minilotes(X, y, tam_lote, rng):
    # Baraja en cada epoca. Sin barajar, el modelo ve siempre los mismos lotes
    # en el mismo orden y el ruido del gradiente deja de ayudar.
    orden = rng.permutation(len(X))
    for i in range(0, len(X), tam_lote):
        idx = orden[i:i + tam_lote]
        yield X[idx], y[idx]


def entrena_red(X, y, n_ocultas=8, epocas=200, tam_lote=32, eta=0.1,
                semilla=20262027, X_val=None, y_val=None):
    rng = np.random.default_rng(semilla)
    n_entradas = X.shape[1]

    # Inicializacion de He. NO a cero: si todos los pesos de una capa valen lo
    # mismo, todas sus neuronas reciben el mismo gradiente y aprenden lo mismo
    # para siempre. Se llama ruptura de simetria.
    W1 = rng.normal(0, np.sqrt(2 / n_entradas), (n_entradas, n_ocultas))
    b1 = np.zeros(n_ocultas)
    W2 = rng.normal(0, np.sqrt(2 / n_ocultas), (n_ocultas, 1))
    b2 = np.zeros(1)

    historia = {"perdida": [], "perdida_val": []}

    for epoca in range(epocas):
        for X_lote, y_lote in minilotes(X, y, tam_lote, rng):
            m = len(X_lote)

            # --- IDA ---
            Z1 = X_lote @ W1 + b1
            A1 = np.maximum(0, Z1)
            Z2 = A1 @ W2 + b2
            A2 = 1 / (1 + np.exp(-Z2))

            # --- VUELTA ---
            # La division por m hace que la tasa de aprendizaje no dependa del
            # tamaño de lote. Si se olvida, pasar de lotes de 32 a lotes de 256
            # multiplica por ocho el paso efectivo y el entrenamiento explota.
            dZ2 = (A2 - y_lote) / m
            dW2 = A1.T @ dZ2
            db2 = dZ2.sum(axis=0)
            dA1 = dZ2 @ W2.T
            dZ1 = dA1 * (Z1 > 0)
            dW1 = X_lote.T @ dZ1
            db1 = dZ1.sum(axis=0)

            # --- ACTUALIZAR ---
            W1 -= eta * dW1;  b1 -= eta * db1
            W2 -= eta * dW2;  b2 -= eta * db2

        pesos = (W1, b1, W2, b2)
        historia["perdida"].append(
            entropia_cruzada_binaria(y, predice(X, pesos)))
        if X_val is not None:
            historia["perdida_val"].append(
                entropia_cruzada_binaria(y_val, predice(X_val, pesos)))

    return (W1, b1, W2, b2), historia


def predice(X, pesos):
    W1, b1, W2, b2 = pesos
    A1 = np.maximum(0, X @ W1 + b1)
    return 1 / (1 + np.exp(-(A1 @ W2 + b2)))

### El problema XOR

XOR es el ejemplo canónico porque **no es separable con una recta**. Es el problema que
Minsky y Papert usaron en 1969 para demostrar que un perceptrón de una capa no podía
resolverlo, y esa demostración congeló la investigación en redes neuronales durante quince
años. La salida fue la capa oculta.

In [ ]:
X_xor = np.array([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y_xor = np.array([[0.], [1.], [1.], [0.]])

# Con UNA unidad oculta la red sigue siendo, en la practica, una recta, y
# con una recta no se separan las dos diagonales de XOR.
#
# Con DOS basta en teoria —es el minimo demostrable— pero el descenso de
# gradiente se queda atascado segun donde caiga la inicializacion. Eso no es
# un fallo del codigo: es el motivo por el que en la practica se pone holgura,
# y es la primera vez en el modulo que aparece un minimo local de verdad.
for n_ocultas in (1, 2, 8):
    pesos, _ = entrena_red(X_xor, y_xor, n_ocultas=n_ocultas,
                           epocas=4000, tam_lote=4, eta=0.5)
    p = predice(X_xor, pesos).ravel()
    print(f"{n_ocultas:>2} unidades ocultas -> {p.round(3)}  "
          f"{'RESUELTO' if np.all((p > 0.5) == y_xor.ravel()) else 'no'}")

In [ ]:
# La frontera de decision, dibujada.
pesos, historia = entrena_red(X_xor, y_xor, n_ocultas=8,
                              epocas=4000, tam_lote=4, eta=0.5)

rejilla = np.linspace(-0.3, 1.3, 300)
GX, GY = np.meshgrid(rejilla, rejilla)
Z = predice(np.c_[GX.ravel(), GY.ravel()], pesos).reshape(GX.shape)

fig, ejes = plt.subplots(1, 2, figsize=(11, 4.2))

mapa = ejes[0].contourf(GX, GY, Z, levels=20, cmap="RdBu_r", vmin=0, vmax=1)
ejes[0].contour(GX, GY, Z, levels=[0.5], colors="black", linewidths=1.5)
ejes[0].scatter(X_xor[:, 0], X_xor[:, 1], c=y_xor.ravel(), cmap="RdBu_r",
                edgecolors="black", s=180, vmin=0, vmax=1, zorder=3)
ejes[0].set_title("La frontera de XOR no es una recta:\nhacen falta dos, y por eso"
                  " hace falta la capa oculta")
ejes[0].set_xlabel("entrada 1");  ejes[0].set_ylabel("entrada 2")
fig.colorbar(mapa, ax=ejes[0], label="probabilidad de clase 1")

ejes[1].plot(historia["perdida"])
ejes[1].set_title("La perdida baja de 0,69 a casi cero")
ejes[1].set_xlabel("epoca");  ejes[1].set_ylabel("entropia cruzada")
ejes[1].axhline(np.log(2), color="crimson", ls="--", lw=1,
                label="log(2) = 0,693, la perdida inicial esperada")
ejes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

---

## 5. La misma red, sobre los clientes de TechStore

Y ahora sobre datos de verdad: los 400 clientes de la UD3, con la partición temporal ya
hecha y la normalización calculada **solo con el entrenamiento**, que es la regla que la
UD3 dejó planteada.

In [ ]:
import os

RUTA_DATOS = os.path.join("datos", "clientes_abandono.csv")
if not os.path.exists(RUTA_DATOS):      # en Colab, desde el espejo publico
    RUTA_DATOS = ("https://raw.githubusercontent.com/RafaSalaEsteve/"
                  "IABD-PIA-notebooks/main/UD5_tensores_redes_neuronales/"
                  "datos/clientes_abandono.csv")

import pandas as pd

tabla = pd.read_csv(RUTA_DATOS)
CARACTERISTICAS = ["recencia_dias", "frecuencia", "monetario", "ticket_medio",
                   "antiguedad_dias", "categorias_distintas", "proporcion_movil",
                   "dias_entre_pedidos", "cat_informatica", "cat_telefonia"]

entrena = tabla[tabla["particion"] == "entrena"]
prueba = tabla[tabla["particion"] == "prueba"]

X_ent = entrena[CARACTERISTICAS].to_numpy(dtype="float64")
X_pru = prueba[CARACTERISTICAS].to_numpy(dtype="float64")
y_ent = entrena[["abandona"]].to_numpy(dtype="float64")
y_pru = prueba[["abandona"]].to_numpy(dtype="float64")

# La media y la desviacion, SOLO del entrenamiento. Es la regla de la UD3.
media, desv = X_ent.mean(axis=0), X_ent.std(axis=0)
desv[desv == 0] = 1.0
X_ent = (X_ent - media) / desv
X_pru = (X_pru - media) / desv

print(f"entrena {X_ent.shape}  abandona el {y_ent.mean() * 100:.1f} %")
print(f"prueba  {X_pru.shape}  abandona el {y_pru.mean() * 100:.1f} %")
print()
print("Esa diferencia de tasa entre las dos particiones es REAL y no es un error:")
print("la particion es temporal, y los clientes recientes se comportan distinto.")

In [ ]:
pesos, historia = entrena_red(X_ent, y_ent, n_ocultas=8, epocas=300,
                              tam_lote=32, eta=0.05,
                              X_val=X_pru, y_val=y_pru)

fig, eje = plt.subplots(figsize=(7, 4))
eje.plot(historia["perdida"], label="entrenamiento")
eje.plot(historia["perdida_val"], label="prueba")
eje.axhline(np.log(2), color="0.6", ls="--", lw=1, label="log(2), sin aprender")
eje.set_title("La curva de entrenamiento baja y la de prueba se da la vuelta:\n"
              "eso es sobreajuste, y se ve sin ninguna teoria")
eje.set_xlabel("epoca");  eje.set_ylabel("entropia cruzada")
eje.legend(fontsize=8)
fig.tight_layout()
plt.show()

print(f"perdida final entrenamiento {historia['perdida'][-1]:.4f}")
print(f"perdida final prueba        {historia['perdida_val'][-1]:.4f}")

In [ ]:
# El punto de referencia de la UD3, y el modelo, con las mismas metricas.
def metricas(y, p, umbral=0.5):
    pred = (p.ravel() >= umbral).astype(int)
    v = y.ravel().astype(int)
    vp = int(((pred == 1) & (v == 1)).sum())
    fp = int(((pred == 1) & (v == 0)).sum())
    fn = int(((pred == 0) & (v == 1)).sum())
    precision = vp / (vp + fp) if vp + fp else 0.0
    exhaustividad = vp / (vp + fn) if vp + fn else 0.0
    f1 = (2 * precision * exhaustividad / (precision + exhaustividad)
          if precision + exhaustividad else 0.0)
    return precision, exhaustividad, f1


def area_roc(y, p):
    # La misma funcion de la UD4: ordenar y acumular. Sin scikit-learn.
    orden = np.argsort(-p.ravel())
    v = y.ravel()[orden]
    positivos, negativos = v.sum(), len(v) - v.sum()
    return float(np.trapezoid(np.cumsum(v) / positivos,
                              np.cumsum(1 - v) / negativos))


p_red = predice(X_pru, pesos)
recencia = prueba["recencia_dias"].to_numpy(dtype="float64")

print(f"{'modelo':32} {'precision':>10} {'exhaust.':>10} {'F1':>7} {'AUC':>7}")
print("-" * 70)
mejor = (0.0, None)
for umbral in (30, 60, 90, 120, 150, 180, 240):
    _, _, f1 = metricas(y_pru, (recencia > umbral).astype(float), 0.5)
    if f1 > mejor[0]:
        mejor = (f1, umbral)
pr, ex, f1 = metricas(y_pru, (recencia > mejor[1]).astype(float), 0.5)
print(f"{'referencia UD3 (recencia > ' + str(mejor[1]) + ' d)':32} "
      f"{pr:>10.3f} {ex:>10.3f} {f1:>7.3f} {area_roc(y_pru, recencia):>7.3f}")

pr, ex, f1 = metricas(y_pru, p_red)
print(f"{'red 10-8-1 en NumPy':32} {pr:>10.3f} {ex:>10.3f} {f1:>7.3f} "
      f"{area_roc(y_pru, p_red):>7.3f}")

> **La red no gana.** No es un fallo del cuaderno: es el resultado, y es el contenido del
> bloque 9 de los apuntes y de todo el cuaderno `UD5_04`. Con 300 filas de entrenamiento y
> diez características ya resumidas a mano en la UD3, no queda representación que
> aprender, y lo único que una capa oculta añade es varianza.
>
> Anótalo y sigue: en el cuaderno 04 se mide con cuidado, y en el proyecto PR5 se ve el
> caso contrario, donde una red arrasa con cualquier regla escrita a mano.

---

## 6. Comprobar contra TensorFlow

La última comprobación del cuaderno, y la que cierra el bloque 5: que los gradientes
escritos a mano son **los mismos** que calcula la diferenciación automática.

In [ ]:
import tensorflow as tf

X_t = tf.constant(X_j, dtype=tf.float64)
y_t = tf.constant(y_j, dtype=tf.float64)
W1_t = tf.Variable(W1);  b1_t = tf.Variable(b1)
W2_t = tf.Variable(W2);  b2_t = tf.Variable(b2)

with tf.GradientTape() as cinta:
    A1_t = tf.nn.relu(X_t @ W1_t + b1_t)
    A2_t = tf.sigmoid(A1_t @ W2_t + b2_t)
    perdida_t = -tf.reduce_mean(y_t * tf.math.log(A2_t)
                                + (1 - y_t) * tf.math.log(1 - A2_t))

grads = cinta.gradient(perdida_t, [W1_t, b1_t, W2_t, b2_t])

print(f"{'parametro':>10} {'diferencia maxima':>20}")
print("-" * 32)
for nombre, mio, suyo in zip(["W1", "b1", "W2", "b2"],
                             [dW1, db1, dW2, db2], grads):
    d = np.abs(mio - suyo.numpy()).max()
    print(f"{nombre:>10} {d:>20.2e}")
    assert d < 1e-10

print()
print("Identicos hasta el error de redondeo.")
print("tf.GradientTape hace EXACTAMENTE lo que acabas de escribir a mano:")
print("diferenciacion automatica en modo inverso, que es la regla de la cadena")
print("evaluada de la salida hacia la entrada.")

---

## Ejercicios

### Ejercicio 1. La activación que falta

Quita la ReLU de `entrena_red` —sustituye `np.maximum(0, Z1)` por `Z1`— y vuelve a
entrenar XOR con 8 unidades ocultas y 4000 épocas. ¿Qué sale? Explica el resultado
citando el apartado 1, y comprueba que la red resultante es equivalente a una capa única
multiplicando `W1 @ W2`.

### Ejercicio 2. La tasa de aprendizaje

Entrena XOR con `eta` en `[0.001, 0.01, 0.1, 0.5, 2.0, 10.0]` y dibuja las seis curvas de
pérdida en la misma figura. Identifica cuál de los cinco patrones del bloque 10 hace cada
una. ¿A partir de qué valor explota, y qué se ve exactamente cuando explota?

### Ejercicio 3. Inicialización

Inicializa `W1` y `W2` a cero y entrena XOR. No aprende. Explica por qué en una celda de
texto, con la palabra *simetría*, y demuéstralo imprimiendo las filas de `W1` después de
cien épocas.

Después prueba con `rng.normal(0, 10, ...)`, que es demasiado grande. Tampoco aprende, por
un motivo **distinto**. Di cuál, y relaciónalo con la figura de las derivadas del
apartado 1.

### Ejercicio 4. Momento

Añade momento al bucle de entrenamiento:

```python
v_W1 = 0.9 * v_W1 + dW1
W1 -= eta * v_W1
```

Compara las curvas de pérdida con y sin momento sobre TechStore, con la misma tasa. ¿Cuántas
épocas hacen falta en cada caso para llegar a la misma pérdida?

### Ejercicio 5. Una capa más

Amplía `entrena_red` a dos capas ocultas. Hay que añadir `W3`, `b3`, un paso más en la ida
y dos más en la vuelta. **Compruébalo con `gradiente_numerico` antes de entrenar**: si el
error relativo pasa de `1e-4`, hay un fallo.

Después entrena sobre TechStore y compara con la red de una capa. ¿Mejora la pérdida de
entrenamiento? ¿Y la de prueba? Relaciona la respuesta con el bloque 9.

### Ejercicio 6. Parada temprana a mano

Modifica `entrena_red` para que guarde una copia de los pesos en la época de menor pérdida
de validación, y devuelva esos en lugar de los últimos. Mide cuánto cambia el F1 sobre
TechStore. Acabas de implementar `EarlyStopping(restore_best_weights=True)`.

---

## Lo que hay que llevarse de aquí

1. **Sin activación no lineal, cincuenta capas son una capa.** Comprobado multiplicando
   las matrices.
2. **La derivada de la sigmoide no pasa de 0,25**, y por eso una red profunda de sigmoides
   no entrena: a las veinte capas el gradiente es $10^{-12}$.
3. **La exactitud no es derivable.** Por eso se optimiza la entropía cruzada y se informa
   de otra cosa.
4. **La pérdida inicial vale $\log(k)$**, y comprobarlo cuesta dos líneas.
5. **`dZ2 = A2 - y` es una cancelación**, y es la razón de que sigmoide y entropía cruzada
   vayan juntas.
6. **El gradiente numérico es la forma de comprobar el analítico.** Lento para entrenar,
   perfecto para depurar.
7. **La inicialización no puede ser cero** —simetría— **ni grande** —saturación—.
8. **La división por el tamaño del lote** es lo que hace que la tasa de aprendizaje no
   dependa de él.
9. **En TechStore la red no gana al punto de referencia**, y eso es el resultado, no un
   error.
10. **`tf.GradientTape` hace exactamente lo que acabas de escribir**, y ahora ya sabes
    qué hay dentro.